In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn import preprocessing
import time
import pickle as pkl
from tensorflow.keras.initializers import HeNormal
import random

seed = 42

random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

print("TensorFlow version:", tf.__version__)
print("Num GPUs Available:", len(tf.config.list_physical_devices('GPU')))

TensorFlow version: 2.19.0
Num GPUs Available: 0


In [11]:
layer_configs = [
    #[16,8],
    [16,8,4],
    [32,16,8],
    [64,32,16],
    [128,64,32],
    [256,128,64]
    #[64,64,32]
]

learning_rates = [0.01]
batch_sizes = [16,32,64]

In [12]:
def log_mae_loss(y_true, y_pred):
    epsilon = 1e-6
    y_true_safe = tf.clip_by_value(y_true, 0.0, 1e6)
    y_pred_safe = tf.clip_by_value(y_pred, 0.0, 1e6)
    
    y_true_log = tf.math.log1p(y_true_safe + epsilon)
    y_pred_log = tf.math.log1p(y_pred_safe + epsilon)
    return tf.reduce_mean(tf.abs(y_true_log - y_pred_log))

In [13]:
df = pd.read_csv('datasets\Independent Scenarios\SRP_50_scenarios\SRP_50_scenario_0.csv')
df['co2_price'] = 0.018


features = df[['0','1','2','3','4','5','6','7','8','9','10','co2_budget', 'co2_price', 'demand_scaling']]
labels = df['ope_cost']

scaler = preprocessing.StandardScaler()
features_scaled = scaler.fit_transform(features)

df_scaled = pd.DataFrame(features_scaled, columns=features.columns)
df_scaled['ope_cost'] = labels.reset_index(drop=True)

X = df_scaled[['0','1','2','3','4','5','6','7','8','9','10','co2_budget', 'co2_price', 'demand_scaling']]
Y = df_scaled['ope_cost']
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)


for layers in layer_configs:
    time_list_tmp = []
    mae_list_tmp = []
    mape_list_tmp = []
    r2_list_tmp = []
    for lr in learning_rates:
        for batch_size in batch_sizes:
            input_shape = X.shape[1:]
            
            model = keras.Sequential()
            model.add(keras.layers.Input(shape = input_shape))
            model.add(keras.layers.Dense(layers[0],activation='relu'))
            for neurons in layers[1:]:
                model.add(keras.layers.Dense(neurons,activation='relu'))
            model.add(keras.layers.Dense(1))  

            model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr), loss=log_mae_loss)

            early_stopping = keras.callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)
            reduce_lr = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=0)
            
            
            
            start_time = time.time()
            history = model.fit(X_train, y_train, 
                    epochs=1000, 
                    batch_size=batch_size, 
                    validation_data=(X_test, y_test), 
                    callbacks=[early_stopping, reduce_lr],
                    verbose=0)
            end_time = time.time()
            training_time = end_time - start_time

            y_pred = model.predict(X_test)
            y_test_exp = y_test
    
            mae = metrics.mean_absolute_error(y_test_exp, y_pred)
            
            mask = y_test_exp != 0
            y_test_no_zero = y_test_exp[mask]
            y_pred_no_zero = y_pred[mask]
            
            mape = metrics.mean_absolute_percentage_error(y_test_no_zero, y_pred_no_zero)
            r2 = metrics.r2_score(y_test_exp, y_pred)
            
            print(f"Config: Layers={layers}, LR={lr}, Batch={batch_size} Train metrics: Train time={training_time} Train loss={min(history.history['loss'])} Test metrics: MAE={mae:.4f}, MAPE={mape:.4f}, R²={r2:.4f}")

        

63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step
Config: Layers=[16, 8, 4], LR=0.01, Batch=16 Train metrics: Train time=221.806978225708 Train loss=0.019634997472167015 Test metrics: MAE=1260.5987, MAPE=0.0216, R²=0.9998
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
Config: Layers=[16, 8, 4], LR=0.01, Batch=32 Train metrics: Train time=84.89249396324158 Train loss=0.020846884697675705 Test metrics: MAE=1249.9088, MAPE=0.0228, R²=0.9998
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
Config: Layers=[16, 8, 4], LR=0.01, Batch=64 Train metrics: Train time=76.52172660827637 Train loss=0.023921893909573555 Test metrics: MAE=1659.1652, MAPE=0.0258, R²=0.9997
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
Config: Layers=[32, 16, 8], LR=0.01, Batch=16 Train metrics: Train time=2167.140741109848 Train loss=0.01338844746351242 Test metrics: MAE=811.7854, MAPE=0.0156, R²=0.9999
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
Config: Layers=[32, 16, 8], LR=0.01, Batch=32 Train metrics: Train time=220.9353232383728 Train loss=0.013125765137

In [12]:
selected_architectures = {
    #0:{'layers': [16,8,4], 'batch size': 32, 'lr':0.01},
    #1:{'layers': [32,16,8], 'batch size': 32, 'lr':0.01},
    2:{'layers': [64,32,16], 'batch size': 16, 'lr':0.01},
    #3:{'layers': [128,64,32], 'batch size': 32, 'lr':0.01},
    #4:{'layers': [256,128,64], 'batch size': 16, 'lr':0.01}
}


n_experiment = 5

for n in range(1,2):
    df = pd.read_csv(f'datasets/SRP 20 scenario/SRP_20_scenario_{n}.csv')
    df['co2_price'] = 0.018
    
    
    features = df[['0','1','2','3','4','5','6','7','8','9','10','co2_budget', 'co2_price', 'demand_scaling']]
    labels = df['ope_cost']

    scaler = preprocessing.StandardScaler()
    features_scaled = scaler.fit_transform(features)

    df_scaled = pd.DataFrame(features_scaled, columns=features.columns)
    df_scaled['ope_cost'] = labels.reset_index(drop=True)
    
    with open(f'Scalers/scaler_20_check_scenario_{n}.pkl', 'wb') as f:
        pkl.dump(scaler, f)
    
    #df_scaled = df_scaled[:10012]
    
    X = df_scaled[['0','1','2','3','4','5','6','7','8','9','10','co2_budget', 'co2_price', 'demand_scaling']]
    Y = df_scaled['ope_cost']
    X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42)
    
    for key, arch in selected_architectures.items():
        
        layers = arch['layers']
        batch_size = arch['batch size']
        lr = arch['lr']
        input_shape = X.shape[1:]
        
        model = keras.Sequential()
        model.add(keras.layers.Input(shape = input_shape))
        model.add(keras.layers.Dense(layers[0],activation='relu', kernel_initializer=keras.initializers.HeNormal()))
        for neurons in layers[1:]:
            model.add(keras.layers.Dense(neurons,activation='relu', kernel_initializer=keras.initializers.HeNormal()))
        model.add(keras.layers.Dense(1))  

        model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr), loss=log_mae_loss)

        early_stopping = keras.callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)
        reduce_lr = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=0)
        
        
        
        start_time = time.time()
        history = model.fit(X_train, y_train, 
                    epochs=1000, 
                    batch_size=batch_size, 
                    validation_data=(X_test, y_test), 
                    callbacks=[early_stopping, reduce_lr],
                    verbose=0)
        end_time = time.time()
        training_time = end_time - start_time

        y_pred = model.predict(X_test)
        y_test_exp = y_test

        mae = metrics.mean_absolute_error(y_test_exp, y_pred)
        
        mask = y_test_exp != 0
        y_test_no_zero = y_test_exp[mask]
        y_pred_no_zero = y_pred[mask]
        
        mape = metrics.mean_absolute_percentage_error(y_test_no_zero, y_pred_no_zero)
        r2 = metrics.r2_score(y_test_exp, y_pred)

        #print(f"Config: dataset = {n} Layers={layers}, LR={lr}, Batch={batch_size} Train metrics: Train time={training_time} Train loss={min(history.history['loss'])} Test metrics: MAE={mae:.4f}, MAPE={mape:.4f}, R²={r2:.4f}")
        print(f"NN_model_15_scenarios_{key}_{n} saved...")
        model.save(f"NN_models/NN_model_20_check_scenarios_{key}_{n}.keras")

94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 895us/step
NN_model_15_scenarios_2_1 saved...
